# RideWise: Customer Analytics & Churn Prediction

### This notebook loads, inspects, merge and fully cleans the datasets:

    ✔️ Correct joins (trip-level)

    ✔️ Clean column naming

    ✔️ Handled meaningful missingness (is_referred)

    ✔️ Fixed datetime consistency (UTC)

    ✔️ Verified duplicate columns logically
    
    ✔️ Initial Feature Engineering
    
    ✔️ Saves a final cleaned CSV for EDA


### Import Required Libraries


In [400]:
import pandas as pd
import numpy as np
import warnings

## Data Loading, Inspection and Merging  

In [402]:
# Drivers data

drivers = pd.read_csv("../data/drivers.csv")

drivers.head()

,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
0,D00000,3.1,SUV,2025-01-20,2025-01-06 18:23:09.312275,Cairo,0.679555
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773


In [403]:
drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   driver_id        5000 non-null   object 
 1   rating           5000 non-null   float64
 2   vehicle_type     5000 non-null   object 
 3   signup_date      5000 non-null   object 
 4   last_active      5000 non-null   object 
 5   city             5000 non-null   object 
 6   acceptance_rate  5000 non-null   float64
dtypes: float64(2), object(5)
memory usage: 273.6+ KB


In [404]:
# Promotions data

promotions = pd.read_csv("../data/promotions.csv")

promotions.head()

,promo_id,promo_name,promo_type,promo_value,start_date,end_date,target_segment,city_scope,ab_test_groups,test_allocation,success_metric
0,P000,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-25,All,Nairobi,['All'],[1.0],Usage Frequency
1,P001,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-22,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate
2,P002,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-16,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",ROI
3,P003,Loyalty Bonus,points,100.0,2025-04-26,2025-05-04,Gold+,Nairobi,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate
4,P004,Loyalty Bonus,points,100.0,2025-04-26,2025-05-15,Gold+,Nairobi,['All'],[1.0],Usage Frequency


In [405]:
promotions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   promo_id         20 non-null     object 
 1   promo_name       20 non-null     object 
 2   promo_type       20 non-null     object 
 3   promo_value      20 non-null     float64
 4   start_date       20 non-null     object 
 5   end_date         20 non-null     object 
 6   target_segment   20 non-null     object 
 7   city_scope       20 non-null     object 
 8   ab_test_groups   20 non-null     object 
 9   test_allocation  20 non-null     object 
 10  success_metric   20 non-null     object 
dtypes: float64(1), object(10)
memory usage: 1.8+ KB


In [406]:
# Customers data

riders = pd.read_csv("../data/riders.csv")

riders.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,NaN
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002


In [407]:
riders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           10000 non-null  object 
 1   signup_date       10000 non-null  object 
 2   loyalty_status    10000 non-null  object 
 3   age               10000 non-null  float64
 4   city              10000 non-null  object 
 5   avg_rating_given  10000 non-null  float64
 6   churn_prob        10000 non-null  float64
 7   referred_by       3053 non-null   object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB


In [408]:
# Sessions data

sessions = pd.read_csv("../data/sessions.csv")

sessions.head()

,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze


In [409]:
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   session_id      50000 non-null  object
 1   rider_id        50000 non-null  object
 2   session_time    50000 non-null  object
 3   time_on_app     50000 non-null  int64 
 4   pages_visited   50000 non-null  int64 
 5   converted       50000 non-null  int64 
 6   city            50000 non-null  object
 7   loyalty_status  50000 non-null  object
dtypes: int64(3), object(5)
memory usage: 3.1+ MB


In [410]:
# Trips data

trips = pd.read_csv("../data/trips.csv")

trips.head()

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 19:27:14+02:05,2024-06-18 19:32:14+02:05,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 09:58:16+02:27,2024-10-05 10:28:16+02:27,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold


In [411]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   trip_id           200000 non-null  object 
 1   user_id           200000 non-null  object 
 2   driver_id         200000 non-null  object 
 3   fare              200000 non-null  float64
 4   surge_multiplier  200000 non-null  float64
 5   tip               200000 non-null  float64
 6   payment_type      200000 non-null  object 
 7   pickup_time       200000 non-null  object 
 8   dropoff_time      200000 non-null  object 
 9   pickup_lat        200000 non-null  float64
 10  pickup_lng        200000 non-null  float64
 11  dropoff_lat       200000 non-null  float64
 12  dropoff_lng       200000 non-null  float64
 13  weather           200000 non-null  object 
 14  city              200000 non-null  object 
 15  loyalty_status    200000 non-null  object 
dtypes: float64(7), objec

## Data Cleaning


In [413]:
# Parse dates/times

# Riders signup date 
if "signup_date" in riders.columns:
    riders["signup_date"] = pd.to_datetime(
        riders["signup_date"],
        errors="coerce",
        utc=True
    )

# Drivers signup date
if "signup_date" in drivers.columns:
    drivers["signup_date"] = pd.to_datetime(
        drivers["signup_date"],
        errors="coerce",
        utc=True
    )

# Trips pickup & dropoff times 
for col in ["pickup_time", "dropoff_time"]:
    if col in trips.columns:
        trips[col] = pd.to_datetime(
            trips[col],
            errors="coerce",
            utc=True
        )


In [414]:
drivers["last_active"] = pd.to_datetime(drivers["last_active"], errors="coerce", utc=True)


In [415]:
# I will drop duplicate columns, from riders and drivers table and keep the column on trip table

riders_clean = riders.drop(columns=["city", "loyalty_status"], errors="ignore")
drivers_clean = drivers.drop(columns=["city"], errors="ignore") 

# I will rename signup dates BEFORE merge so they don't become _x/_y

riders_clean = riders_clean.rename(columns={"signup_date": "rider_signup_date"})
drivers_clean = drivers_clean.rename(columns={"signup_date": "driver_signup_date"})

In [416]:
# I will create a referral flag from the riders table and drop referred_by column

riders_clean["is_referred"] = riders_clean["referred_by"].notna().astype(int)

riders_clean = riders_clean.drop(columns=["referred_by"])

In [501]:
master_df['age'] = master_df['age'].astype(int)

## Merge Trip-level Datasets

#### I want one big table (Master_df) where:

 * Each row = one trip
 * That row will know:
    * who the rider is
    * what city they’re in
    * what the driver is like
    * what the fare, time, weather, etc. were

So instead of jumping between 3–4 tables, I will explore one table.

That’s what a Master_df table is.

In [419]:
master_df = (trips
              .merge(riders_clean, on="user_id", how="left")
              .merge(drivers_clean, on="driver_id", how="left"))

In [420]:
master_df.head()

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,...,rider_signup_date,age,avg_rating_given,churn_prob,is_referred,rating,vehicle_type,driver_signup_date,last_active,acceptance_rate
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,...,2023-09-20 00:00:00+00:00,37.629694,4.3,0.294348,0,4.1,Sedan,2024-07-21 00:00:00+00:00,2025-03-21 22:47:26.558938+00:00,0.549628
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,...,2024-08-30 00:00:00+00:00,37.051088,4.7,0.105990,1,4.9,SUV,2023-05-06 00:00:00+00:00,2025-04-12 08:36:21.207528+00:00,0.629250
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,...,2023-12-03 00:00:00+00:00,44.708377,3.8,0.169081,1,4.7,Sedan,2023-08-09 00:00:00+00:00,2025-04-19 05:19:12.026949+00:00,0.990000
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14+00:00,2024-06-18 17:27:14+00:00,29.819554,...,2024-04-15 00:00:00+00:00,23.229117,4.3,0.211581,1,3.9,Sedan,2023-04-22 00:00:00+00:00,2025-02-08 22:15:23.685705+00:00,0.247121
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16+00:00,2024-10-05 08:01:16+00:00,-1.676479,...,2025-01-19 00:00:00+00:00,34.884552,5.0,0.720780,0,4.2,Sedan,2024-08-22 00:00:00+00:00,2025-04-13 11:24:50.358526+00:00,0.611031


In [421]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 26 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   trip_id             200000 non-null  object             
 1   user_id             200000 non-null  object             
 2   driver_id           200000 non-null  object             
 3   fare                200000 non-null  float64            
 4   surge_multiplier    200000 non-null  float64            
 5   tip                 200000 non-null  float64            
 6   payment_type        200000 non-null  object             
 7   pickup_time         200000 non-null  datetime64[ns, UTC]
 8   dropoff_time        200000 non-null  datetime64[ns, UTC]
 9   pickup_lat          200000 non-null  float64            
 10  pickup_lng          200000 non-null  float64            
 11  dropoff_lat         200000 non-null  float64            
 12  dropoff_lng     

## Examine dataset duplicate values

In [423]:
# Check for duplicate values

print("\n📌 Dataset Columns:")
master_df.duplicated().sum()


📌 Dataset Columns:


0

In [424]:
# Numerical Statisticcal Analysis

master_df.describe()

,fare,surge_multiplier,tip,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,age,avg_rating_given,churn_prob,is_referred,rating,acceptance_rate
count,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000
mean,15.401285,1.141499,0.469566,11.849600,23.924133,11.849589,23.924173,35.140091,4.461349,0.285692,0.305065,4.173626,0.694781
std,6.163199,0.255362,1.100545,13.362151,14.577572,13.362229,14.577642,9.546061,0.428432,0.159011,0.460436,0.594107,0.185828
min,2.970000,1.000000,0.000000,-1.786360,2.879224,-1.833220,2.830979,18.000000,2.600000,0.002934,0.000000,3.100000,0.100000
25%,11.000000,1.000000,0.000000,-1.172683,3.496574,-1.172868,3.497195,28.308293,4.200000,0.161546,0.000000,3.700000,0.564874
50%,14.130000,1.000000,0.000000,6.525574,31.238814,6.525235,31.239118,35.021160,4.500000,0.265622,0.000000,4.200000,0.696067
75%,18.350000,1.200000,0.400000,29.934766,36.703772,29.935056,36.704067,41.673472,4.800000,0.388132,1.000000,4.700000,0.835793
max,82.740000,3.800000,21.860000,30.544251,37.317090,30.592457,37.364817,70.000000,5.000000,0.913302,1.000000,5.000000,0.990000


In [425]:
# Categorical Statisticcal Analysis

master_df.describe(include="object")

,trip_id,user_id,driver_id,payment_type,weather,city,loyalty_status,vehicle_type
count,200000,200000,200000,200000,200000,200000,200000,200000
unique,200000,10000,5000,3,4,3,4,4
top,T000000,R08152,D03093,Card,Sunny,Cairo,Bronze,Sedan
freq,1,42,67,100326,120151,67436,121252,97933


### Examine columns for disparities

In [427]:
# Examine the categorical columns of the data

categorical = master_df.select_dtypes(include="object")

categorical.head()

,trip_id,user_id,driver_id,payment_type,weather,city,loyalty_status,vehicle_type
0,T000000,R05207,D00315,Card,Foggy,Nairobi,Bronze,Sedan
1,T000001,R09453,D03717,Card,Sunny,Lagos,Gold,SUV
2,T000002,R00567,D02035,Card,Cloudy,Nairobi,Bronze,Sedan
3,T000003,R09573,D02657,Mobile Money,Cloudy,Cairo,Bronze,Sedan
4,T000004,R03446,D01026,Card,Sunny,Nairobi,Gold,Sedan


In [428]:
# Examine the unique features in the categorical columns of the data

# List of columns to ignore
id_columns = ["trip_id", "user_id", "driver_id", "last_active"]

for col in categorical.columns:
    if col not in id_columns:
        print(f"\nColumn: {col}")
        print("Unique values:", categorical[col].unique())
        print("Count of unique values:", categorical[col].nunique())


Column: payment_type
Unique values: ['Card' 'Mobile Money' 'Cash']
Count of unique values: 3

Column: weather
Unique values: ['Foggy' 'Sunny' 'Cloudy' 'Rainy']
Count of unique values: 4

Column: city
Unique values: ['Nairobi' 'Lagos' 'Cairo']
Count of unique values: 3

Column: loyalty_status
Unique values: ['Bronze' 'Gold' 'Silver' 'Platinum']
Count of unique values: 4

Column: vehicle_type
Unique values: ['Sedan' 'SUV' 'Motorcycle' 'Luxury']
Count of unique values: 4


In [429]:
### Check for Missing Values

In [430]:
# To check missing values per column
print("\n📌 Missing Values per Column")
print(master_df.isnull().sum())


📌 Missing Values per Column
trip_id               0
user_id               0
driver_id             0
fare                  0
surge_multiplier      0
tip                   0
payment_type          0
pickup_time           0
dropoff_time          0
pickup_lat            0
pickup_lng            0
dropoff_lat           0
dropoff_lng           0
weather               0
city                  0
loyalty_status        0
rider_signup_date     0
age                   0
avg_rating_given      0
churn_prob            0
is_referred           0
rating                0
vehicle_type          0
driver_signup_date    0
last_active           0
acceptance_rate       0
dtype: int64


### Initial Feature Engineering

In [432]:
# Calculate Revenue

master_df["trip_revenue"] = (master_df["fare"] * master_df["surge_multiplier"]) + master_df["tip"]

In [433]:
# Calculate trip duration in minutes (integer)

master_df["trip_duration_mins"] = (
    (master_df["dropoff_time"] - master_df["pickup_time"])
    .dt.total_seconds() / 60
).round().astype(int)

In [503]:
# Compute rider age groups

def age_group(age):
    if age <= 25:
        return 'Emerging Adults'
    elif age <= 35:
        return 'Young Professionals'
    elif age <= 45:
        return 'Established Adults'
    elif age <= 55:
        return 'Experienced Adults'
    else:
        return 'Mature Adults'


master_df['rider_age_group'] = master_df['age'].apply(age_group)
master_df[['age','rider_age_group']].head(3).reset_index(drop=True)

,age,rider_age_group
0,37,Established Adults
1,37,Established Adults
2,44,Established Adults


In [435]:
# Extract pickup_hour, pickup_date, pickup_day, pickup_month, pickup_year, pickup_period

master_df['pickup_hour'] = master_df['pickup_time'].dt.hour
master_df['pickup_date'] = master_df['pickup_time'].dt.date
master_df['pickup_day'] = master_df['pickup_time'].dt.day_name()
master_df['pickup_month'] = master_df['pickup_time'].dt.month_name()
master_df['pickup_year'] = master_df['pickup_time'].dt.year

def pickup_period(hour):
    if 0 <= hour < 4:
        return 'Late Night'
    elif 4 <= hour < 8:
        return 'Early Morning'
    elif 8 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 16:
        return 'Afternoon'
    elif 16 <= hour < 20:
        return 'Evening'
    else:
        return 'Night'

master_df['pickup_period'] = master_df['pickup_time'].apply(lambda x: pickup_period(x.hour))

In [436]:
# Extract drop_off_day, and drop_off_date

master_df['drop_off_day'] = (
    master_df['dropoff_time'].dt.day_name()
)

master_df['drop_off_date'] = (
    master_df['dropoff_time'].dt.floor('D')
)

### Final dataset check

In [505]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 37 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   trip_id             200000 non-null  object             
 1   user_id             200000 non-null  object             
 2   driver_id           200000 non-null  object             
 3   fare                200000 non-null  float64            
 4   surge_multiplier    200000 non-null  float64            
 5   tip                 200000 non-null  float64            
 6   payment_type        200000 non-null  object             
 7   pickup_time         200000 non-null  datetime64[ns, UTC]
 8   dropoff_time        200000 non-null  datetime64[ns, UTC]
 9   pickup_lat          200000 non-null  float64            
 10  pickup_lng          200000 non-null  float64            
 11  dropoff_lat         200000 non-null  float64            
 12  dropoff_lng     

### Save the Cleaned Dataset

In [440]:
# Save cleaned data 

master_df_cleaned = master_df.copy()

master_df_cleaned.to_csv(r"C:\Users\pehlu\OneDrive\projects\RideWise\Data\RideWise_cleaned_df.csv", index=False)